In [0]:
%run ../config/config_init

In [0]:
domain_name = create_widget("domain_name", "archive")
job_id = create_widget("job_id")
job_run_id = create_widget("job_run_id")
task_run_id = create_widget("task_run_id")
orchestrator_job_id = create_widget("orchestrator_job_id")
orchestrator_job_run_id = create_widget("orchestrator_job_run_id")

domain_name = dbutils.widgets.get("domain_name")
job_id = dbutils.widgets.get("job_id")
job_run_id = dbutils.widgets.get("job_run_id")
task_run_id = dbutils.widgets.get("task_run_id")
orchestrator_job_id = dbutils.widgets.get("orchestrator_job_id")
orchestrator_job_run_id = dbutils.widgets.get("orchestrator_job_run_id")

In [0]:
rows = (
    spark.table(f"{catalog_name}.{schema_config}.configurations_table")
         .filter(f"domain_name = '{domain_name}' AND medallion_layer = 'bronze'")
         .collect()
)

for row in rows:
    try:
        print(f"Moving files from {row.source} to {row.target}...")

        files = dbutils.fs.ls(row.source)

        for f in files:
            if f.isFile():
                dbutils.fs.cp(f.path, row.target + f.name)
                dbutils.fs.rm(f.path)
                print(f"Moved {f.name} to {row.target}")
        else:
            if not files:
                print(f"No files found in {row.source}.")
    except Exception as e:
        print(f"Error moving files: {e}.")